# E1.1 · Why point-in-time control testing fails for AI

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.0 · Start here — what AI governance means](https://spbreed.github.io/cyber-commons/lessons/E1.0.html)**.

| | |
|---|---|
| Tools used | promptfoo |

## What this lesson is

**What it covers.** Change a prompt and show the control evidence going stale in real time.

**Why a security engineer needs it.** An annual review certifies nothing about a system that changed on Tuesday. The control it builds is: continuous assurance; control effectiveness redefined for probabilistic systems.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You tested the control in March and signed the assertion. The prompt changed in April, the model in May, and the tool scope in June. The assertion is still on file and has not described anything real since the day it was written.

> **At CyberTravels.** The control test that passed in March described a CyberTravels with no payments scope, no repository access and no vector store. Nothing about it was wrong; everything about it is stale.

## 2 · The framework

```
   march      test the control, sign the assertion
   april      the prompt changes
   may        the model version changes
   june       the tool scope changes
   december   the assertion is still on file

   point-in-time assurance for a system that changes between tests
   describes a system that no longer exists
```

Classical control testing has a simple shape: a control is designed, an auditor
tests it once or twice a year, and a passing test is recorded for the period.

That works when the thing being tested changes only through a process that
generates evidence. For an agent, the four things that change its behaviour are:

- the **model version** — changed by your provider, possibly without notice,
- the **prompt** — edited in a console,
- the **tool manifest** — a config change,
- the **approval settings** — a toggle in an admin UI.

None of them is a code change. None generates a change record. All of them
invalidate the conditions the control was tested under.

The honest consequence is that a control tested six months ago is not passing —
it is **unevidenced**, which is a third state most GRC tooling cannot represent.
Introducing that third state is the whole of this lesson.

## 3 · The control — a freshness window per control, derived from drift

The window is not an audit-calendar choice. It comes from **how fast the thing the control tests actually changes.**

## 4 · What replaces the annual test, as a skill

If the window is short, something has to re-run inside it, and that something is an attestation: collect each control's verdict, resolve every evidence pointer, compute drift against the image digest and the tool manifest, and sign the result. Two rules in the procedure are the whole difference between an attestation and a slide — a missing verdict is not a pass, and a capped verdict does not get raised because the other evidence looked good. This is the file in this repository:

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/attestation/attestation-signer-lifecycle/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: attestation-signer-lifecycle
description: >-
  Bundle control evidence, compute verdicts, and produce a signed verifiable
  attestation for a deployment, re-attesting on drift. Use to emit or verify
  a control attestation, to sign evidence into an in-toto envelope, or to
  decide whether drift requires re-attestation.
allowed-tools: Bash, Read, Write
---

# Attestation Signer Lifecycle

**Controls:** All — the artifact

## Use the existing framework

Do not invent a format. Wrap the evidence in an **in-toto style statement**
inside a signed envelope, with the subject naming the deployment and its
digests, and a typed predicate carrying the control verdicts. Express the
predicate body in an assessment-results vocabulary so the verdicts map to
control catalogues and an auditor can consume them.

The roles are worth naming explicitly: this skill set is the **attester**
producing evidence, a separate service is the **verifier** appraising it
against reference values, and the deployment gate is the **relying party**
applying policy. Keep them separate — an attester that also decides whether it
passed is not an attestation.

## Procedure

1. **Collect every sub-skill verdict.** A missing verdict is not a pass. If a
   skill did not run, the control is `UNKNOWN` and the attestation says so.
2. **Apply the confidence ceilings.** Sandbox-egress and injection-screening are
   capped at `PARTIAL` by their own skills; the signer must refuse to raise them.
3. **Resolve every evidence pointer.** A URI that does not resolve is a broken
   attestation, not a cosmetic issue.
4. **Attach framework mappings**, so one artefact answers several catalogues.
5. **Compute drift.** Compare image digest, configuration hashes and tool
   description hashes against the previous attestation. Any change triggers
   re-attestation — the tool-description hash specifically defends against a
   server mutating a tool after approval.
6. **Sign**, and store alongside the image or in a transparency log.

## Output contract

```json
{
  "subject": [{"name": "deployment_id", "digest": {"image": "sha256:…", "repo": "str"}}],
  "predicate_type": "str",
  "predicate": {
    "deployment_id": "str", "evaluated_at": "str", "evaluator_version": "str",
    "controls": [
      {"id": "C1_default_deny_least_privilege",
       "verdict": "PASS|FAIL|PARTIAL|UNKNOWN",
       "confidence": "HIGH|PARTIAL",
       "evidence": [{"skill": "str", "uri": "str", "hash": "str"}],
       "findings": ["str"]}
    ],
    "framework_mappings": {"owasp_llm": ["str"], "owasp_agentic": ["str"],
                           "atlas": ["str"], "nist_ai_rmf": ["str"], "iso_42001": ["str"]},
    "drift": {"since": "str", "changed": ["str"]}
  },
  "signatures": [{"keyid": "str", "sig": "str"}]
}
```

## Failure modes

- **Reading a missing attestation as a pass.** The relying party must **fail
  closed**: a signed file can be deleted, and absence is not evidence.
- **Raising a capped verdict** because the other evidence looked good.
- **Signing without resolving evidence pointers**, which produces a
  tamper-evident document full of dead links.
- **Attesting once.** Without drift-triggered re-attestation the artefact
  describes a deployment that may no longer exist.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

The skill loads and reports its shape, and its failure modes are the governance lesson stated as engineering: the relying party must fail closed on a missing attestation, because reading absence as a pass is exactly the annual-test habit arriving in a new format. Drift against the digest and the manifest is what re-triggers it, not the calendar.

## Your turn

Pick your three most important AI controls and set a freshness window for each from the observed change rate of what it tests. Then recompute your posture. The number will drop, and it will be the first honest one you have had.

---

**Next → [E1.2 · Building the AI and agent inventory](https://spbreed.github.io/cyber-commons/lessons/E1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*